# 07 — STEP 4: Endpoint smoke test (no mocks)

Hits every endpoint in the API contract against a **running server** and shows the
real JSON. Uses `urllib` (stdlib) so UTF-8 (Kannada) is handled cleanly.

**Start the server first** in a terminal (seeded local DB):

```
DATABASE_URL=sqlite:///./pravah.db AUTH_DISABLED=true \
  python -m uvicorn app.main:app --host 127.0.0.1 --port 8000
```

Wait until `/health` shows `models_loaded: true` (LaBSE load takes ~30s), then run
the cells below.

In [ ]:
import json, urllib.request, urllib.error
B = "http://127.0.0.1:8000"

def get(path):
    return json.load(urllib.request.urlopen(B + path, timeout=30))

def post(path, body):
    req = urllib.request.Request(B + path, data=json.dumps(body).encode("utf-8"),
                                 headers={"Content-Type": "application/json"}, method="POST")
    return json.load(urllib.request.urlopen(req, timeout=60))

def show(title, obj):
    print(title)
    print(json.dumps(obj, ensure_ascii=False, indent=2)[:1200])
    print()

### /health

In [ ]:
show("/health", get("/health"))

### /api/summary

In [ ]:
show("/api/summary", get("/api/summary"))

### /api/events + /api/events/{id}

In [ ]:
events = get("/api/events?limit=2")
show("/api/events?limit=2", events)
eid = events[0]["id"]
show(f"/api/events/{eid}", get(f"/api/events/{eid}"))

### /api/triage — English AND Kannada

In [ ]:
show("/api/triage (EN)", post("/api/triage",
     {"text": "Major accident with overturned truck blocking two lanes on Hosur Road"}))
show("/api/triage (KN)", post("/api/triage",
     {"text": "ಮೈಸೂರು ರಸ್ತೆಯಲ್ಲಿ ಭಾರಿ ಅಪಘಾತ, ಎರಡು ಲೇನ್ ಬಂದ್ ಆಗಿದೆ"}))

### /api/predict

In [ ]:
show("/api/predict", post("/api/predict", {
    "event_cause": "accident", "veh_type": "car", "corridor": "Hosur Road",
    "zone": "south", "event_type": "unplanned", "requires_road_closure": True,
    "hour": 18, "dow": 0}))

### /api/predictions/corridors · /api/hotspots · /api/alerts

In [ ]:
show("/api/predictions/corridors?horizon=1h (top3)", get("/api/predictions/corridors?horizon=1h")[:3])
hs = get("/api/hotspots?horizon=now")
print("hotspots: type", hs["type"], "| n_features", len(hs["features"]))
show("hotspots feature[0]", hs["features"][0])
show("/api/alerts (first 2)", get("/api/alerts")[:2])

### /api/simulate · /api/analytics

In [ ]:
sim = post("/api/simulate", {"trigger": "accident", "corridor": "Hosur Road", "duration_hours": 2})
sim["routes_geojson"] = f"<FeatureCollection {len(sim['routes_geojson'].get('features', []))} features>"
show("/api/simulate", sim)
show("/api/analytics", get("/api/analytics"))

### /api/stream (SSE) — read the first event

In [ ]:
import urllib.request
with urllib.request.urlopen(B + "/api/stream", timeout=5) as r:
    print("first SSE line:", r.readline().decode().strip())